In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [5]:
# A. EXTRACT
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Total Baris Transaksi: {df_trx.count()}")
df_trx.printSchema()

print(f"Total Baris Produk: {df_produk.count()}")
df_produk.printSchema()

print(f"Total Baris Ulasan: {df_ulasan.count()}")
df_ulasan.printSchema()

Total Baris Transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Total Baris Produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Total Baris Ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
from pyspark.sql.functions import col, when

# B. TRANSFORM - Penggabungan
# 1. Transaksi ke Produk (Inner Join karena setiap transaksi pasti punya produk)
df_etl = df_trx.join(df_produk, on="product_id", how="inner")

# 2. Transaksi ke Ulasan (Left Join karena tidak semua transaksi punya ulasan)
df_etl = df_etl.join(df_ulasan, on="order_id", how="left")

# 3. Kolom total_pendapatan
df_etl = df_etl.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

# C. TRANSFORM - Penanganan Data Kosong
# Tambahkan kolom ada_ulasan sebelum mengisi null
df_etl = df_etl.withColumn("ada_ulasan", when(col("rating").isNotNull(), True).otherwise(False))

# Mengisi rating kosong (null) dengan 0
df_etl = df_etl.fillna({"rating": 0})

df_etl.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
+--------+----------+---

In [8]:
# D. LOAD
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Membuat direktori HDFS jika belum ada
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

# Menulis Parquet terpartisi
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(output_path)

# Verifikasi struktur folder
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Verifikasi total count data tersimpan
df_final = spark.read.parquet(output_path)
print("Data tersimpan utuh. Jumlah baris:", df_final.count())

-rw-r--r--   3 dimas supergroup          0 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - dimas supergroup          0 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 dimas supergroup      16364 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-a26ddb36-6342-4a7d-babf-c1784a361b52.c000.snappy.parquet
drwxr-xr-x   - dimas supergroup          0 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 dimas supergroup      11134 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-a26ddb36-6342-4a7d-babf-c1784a361b52.c000.snappy.parquet
drwxr-xr-x   - dimas supergroup          0 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 dimas supergroup      10393 2026-09-24 17:39 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-a26ddb36-6342-4a7d-babf-c1784a361b52.c000.snappy.parquet
drwxr-xr-x   - dimas sup

In [9]:
from pyspark.sql.functions import avg, round

# Mengkonversi boolean ke integer (True=1, False=0), 
# lalu di-average untuk mendapat rasio persentase ulasan.
df_insight = df_final.withColumn("ada_ulasan_int", col("ada_ulasan").cast("int")) \
    .groupBy("kategori") \
    .agg(round(avg("ada_ulasan_int") * 100, 2).alias("persentase_diulas_pct")) \
    .orderBy("persentase_diulas_pct")

df_insight.show()

+------------+---------------------+
|    kategori|persentase_diulas_pct|
+------------+---------------------+
|     Makanan|                68.84|
|   Kesehatan|                68.89|
|     Fashion|                70.14|
|Rumah Tangga|                70.55|
|  Elektronik|                70.66|
+------------+---------------------+



Interpretasi Bisnis:
Mengetahui kategori mana yang minim ulasan (feedback) sangat krusial. Seperti halnya membangun manajemen operasional kafe, ulasan bertindak sebagai social proof (bukti sosial) yang menumbuhkan kepercayaan pelanggan baru. Jika tim marketing mengetahui kategori produk X memiliki persentase ulasan terendah, mereka bisa memprioritaskan pembuatan promo, memberikan poin tambahan (insentif), atau campaign khusus agar pelanggan yang membeli produk di kategori tersebut mau meninggalkan rating, sehingga ujungnya berimbas pada konversi penjualan yang meningkat.